# 🍓 草莓偵測與座標紀錄系統
---
此程式將執行影片偵測，並將每一顆偵測到的草莓位置（Bounding Box）詳細記錄至 `.txt` 檔案中。

### 📋 功能特色：
- **自動路徑檢索**：自動抓取最新的模型。
- **環境自癒**：自動修正路徑以加載 `agent_tools`。

In [2]:
import os, sys, glob, cv2, time
import numpy as np
from ultralytics import YOLO
from tqdm.notebook import tqdm

# 🛠️ 修正路徑以載入 agent_tools (若位於子目錄執行)
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path: sys.path.append(ROOT_DIR)

try:
    from agent_tools.yolo_utils import find_latest_model_path
except ImportError:
    # 如果匯入失敗，定義後備方案
    def find_latest_model_path(base_dir=None):
        if base_dir is None: base_dir = os.path.join(ROOT_DIR, 'runs', 'detect')
        search_patterns = [os.path.join(base_dir, 'train*'), os.path.join(base_dir, 'exp*')]
        dirs = []
        for pattern in search_patterns: dirs.extend(glob.glob(pattern))
        valid_dirs = [d for d in dirs if os.path.exists(os.path.join(d, 'weights', 'best.pt'))]
        if valid_dirs:
            latest_dir = max(valid_dirs, key=os.path.getmtime)
            return os.path.join(latest_dir, 'weights', 'best.pt')
        return os.path.join(ROOT_DIR, 'best_weights', 'best.pt')

# ==========================================
# ⚙️ 參數設定
# ==========================================
SOURCE_LIST = [
    os.path.join(ROOT_DIR, "測試模型用/模擬_1.mp4"),
]

CONF_THRESHOLD = 0.5
SAVE_TXT = True
SAVE_VIDEO = True
OUTPUT_DIR = "偵測結果紀錄"

def run_detection_and_log(sources):
    # 1. 初始化模型 (指定基底目錄)
    model_path = find_latest_model_path(os.path.join(ROOT_DIR, 'runs', 'detect'))
    print(f"🎯 載入模型: {model_path}")
    model = YOLO(model_path)
    
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        
    date_str = time.strftime("%Y%m%d")
    
    for video_source in sources:
        if not os.path.exists(video_source):
            print(f"❌ 找不到影片: {video_source}"); continue
            
        video_name = os.path.basename(video_source)
        base_name = os.path.splitext(video_name)[0]
        
        txt_path = os.path.join(OUTPUT_DIR, f"{base_name}_{date_str}_coords.txt")
        video_out_path = os.path.join(OUTPUT_DIR, f"{base_name}_{date_str}_output.mp4")
        
        cap = cv2.VideoCapture(video_source)
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        video_writer = None
        if SAVE_VIDEO:
            video_writer = cv2.VideoWriter(video_out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
            
        print(f"🚀 開始處理: {video_name}")
        
        with open(txt_path, 'w', encoding='utf-8') as f_log:
            f_log.write("frame,track_id,x1,y1,x2,y2,confidence,class\n")
            
            pbar = tqdm(total=total_frames, desc=f"偵測中 {video_name[:10]}")
            frame_idx = 0
            
            while cap.isOpened():
                success, frame = cap.read()
                if not success: break
                frame_idx += 1
                pbar.update(1)
                
                results = model.track(frame, persist=True, conf=CONF_THRESHOLD, verbose=False)
                
                if results[0].boxes.id is not None:
                    boxes = results[0].boxes.xyxy.cpu().numpy()
                    ids = results[0].boxes.id.cpu().numpy().astype(int)
                    confidences = results[0].boxes.conf.cpu().numpy()
                    classes = results[0].boxes.cls.cpu().numpy().astype(int)
                    
                    for i in range(len(ids)):
                        x1, y1, x2, y2 = boxes[i]
                        f_log.write(f"{frame_idx},{ids[i]},{x1:.2f},{y1:.2f},{x2:.2f},{y2:.2f},{confidences[i]:.4f},{classes[i]}\n")
                
                if SAVE_VIDEO:
                    video_writer.write(results[0].plot())
            
            pbar.close()
            
        cap.release()
        if video_writer: video_writer.release()
        print(f"✅ {video_name} 處理完成，紀錄儲存於: {txt_path}")

if __name__ == "__main__":
    run_detection_and_log(SOURCE_LIST)


🎯 載入模型: d:\銘澄專區\畢業專題工作區\runs\detect\exp2b_1a_img800\weights\best.pt
🚀 開始處理: 模擬_1.mp4


偵測中 模擬_1.mp4:   0%|          | 0/192 [00:00<?, ?it/s]

✅ 模擬_1.mp4 處理完成，紀錄儲存於: 偵測結果紀錄\模擬_1_20260505_coords.txt
